## The Concept of Persistence in LangGraph


In [21]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langgraph.checkpoint.memory import InMemorySaver

In [22]:
load_dotenv()


True

In [23]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


In [24]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [25]:
def generate_joke(state: JokeState):
    prompt = f"Generate a joke on the topic {state["topic"]}"
    response = llm.invoke(prompt).content
    return {'joke': response}
def generate_explanation(state: JokeState):
    prompt = f"Generate an explanation on the joke {state["joke"]}"
    response = llm.invoke(prompt).content
    return {'explanation': response}

In [26]:
#graph
graph = StateGraph(JokeState)
graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)



In [27]:
config1 = {"configurable":{"thread_id": "1"}}
workflow.invoke({'topic':'pizza'},config=config1)
workflow.get_state(config1)
list(workflow.get_state_history(config1))

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 15.606876584s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

In [ ]:
config2 = {"configurable":{"thread_id": "2"}}
workflow.invoke({'topic':'pasta'},config=config2)
print(workflow.get_state(config1))
print(workflow.get_state(config2))

## Time Travel


In [ ]:
workflow.get_state({'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f116102-5e95-6605-bfff-527850cc13eb'}}) # Every checkpoint has a corresponding checkpoint_id to it


StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f116102-5e95-6605-bfff-527850cc13eb'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [ ]:
workflow.invoke(None,{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f116102-5e95-6605-bfff-527850cc13eb'}})


EmptyInputError: Received no input for __start__

In [ ]:
list(workflow.get_state_history(config1))


[StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza chef get fired?\n\nBecause he kept making too many *mis-steaks* and couldn't *knead* the dough!"}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11664a-36c2-67ae-8001-d90da96eddfd'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-03-02T18:21:31.951075+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f116649-e7ec-6a1e-8000-7b88b736c0a4'}}, tasks=(PregelTask(id='243a97d6-0911-578a-7979-0bda25c2d3d1', name='generate_explanation', path=('__pregel_pull', 'generate_explanation'), error='ChatGoogleGenerativeAIError("Error calling model \'gemini-2.5-flash\' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {\'error\': {\'code\': 429, \'message\': \'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.de

## Updating State


In [ ]:
workflow.update_state({'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f116102-5e95-6605-bfff-527850cc13eb'}},{'topic':'samosa'})


{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f11664e-5ef9-6c8f-8000-74f98dcc91d6'}}

In [ ]:
list(workflow.get_state_history(config1))


[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11664e-5ef9-6c8f-8000-74f98dcc91d6'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-03-02T18:23:23.542221+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f116102-5e95-6605-bfff-527850cc13eb'}}, tasks=(PregelTask(id='051843d9-89f8-c91a-da79-5f06c2c666d4', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza chef get fired?\n\nBecause he kept making too many *mis-steaks* and couldn't *knead* the dough!"}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11664a-36c2-67ae-8001-d90da96eddfd'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, cre

In [ ]:
workflow.invoke(None,{'configurable': {'thread_id': '1', 'checkpoint_ns': '',  'checkpoint_id': '1f116127-5ca7-6a5e-8000-77528d277694'}})


EmptyInputError: Received no input for __start__

## Fault Tolerance using Persistence


In [ ]:
import time

# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str
    step3: str

def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(30)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"step3": "done"}

In [ ]:
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.add_edge(START, "step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()

graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
❌ Kernel manually interrupted (crash simulated).


In [ ]:
graph.get_state({"configurable": {"thread_id": 'thread-1'}})


StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f116650-f64e-6fd5-8001-495047a30462'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-03-02T18:24:33.097711+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f116650-f64c-650e-8000-90da7de91c7c'}}, tasks=(PregelTask(id='1cfa6056-6f63-6861-a4e0-0801ae74db94', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [ ]:
# 6. Re-run to show fault-tolerant resume
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
✅ Step 3 executed

✅ Final State: {'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}
